# DF-B Ders Kitabı: Zengin ama Kısa Pencere

Bu notebook, [`df_a_ders_kitabi.ipynb`](df_a_ders_kitabi.ipynb)'nin
KARDEŞ dosyasıdır. DF-A'yı daha önce okuduysan, oradaki temel kavramları
(zaman serisi, korelasyon, log-değişim) burada TEKRAR öğretmeyeceğiz —
yalnızca kısaca hatırlatıp DF-B'ye ÖZGÜ yeni konulara (BETAM, ENAG, az
gözlemle çalışma) odaklanacağız.

**Notebook'un ele aldığı veri seti:** `df_b_zengin_2024_bugun_v2.csv`
(kısaca "DF-B" diyeceğiz).


---
## Bölüm 1 — Bu Veri Seti Ne, DF-A'dan Farkı Ne?

DF-B, 2024-01'den bugüne uzanan (yalnızca **30 ay**) ama DF-A'da HİÇ
olmayan iki ek kaynağı içeren bir tablo:

- **BETAM** (Bahçeşehir Üniversitesi'ne bağlı bir araştırma merkezi) —
  ikinci el otomobil piyasasının fiyatını, ilanda kalma süresini ve satış
  oranını ölçen aylık raporlar yayımlıyor.
- **ENAG** (Enflasyon Araştırma Grubu) — TÜİK'in resmi enflasyon
  rakamına BAĞIMSIZ bir alternatif ölçüm sunan bir grup.

Bu iki kaynağın ikisi de yalnızca 2024-01'den itibaren projeye dahil
edilebildi (o tarihten önceki veriye erişilemedi) — bu yüzden DF-B, DF-A'ya
göre çok daha KISA bir zaman penceresi kapsıyor.

| | DF-A | DF-B |
|---|---|---|
| Zaman aralığı | 2018-01 → 2026-06 (**102 ay**) | 2024-01 → 2026-06 (**30 ay**) |
| BETAM (fiyat, ilan süresi, satış oranı) | ❌ Yok | ✅ Var |
| ENAG (alternatif enflasyon) | ❌ Yok | ✅ Var |
| Özet | **Uzun ama dar** | **Kısa ama geniş** |

Şimdi veriyi yükleyip ilk bakışı atalım.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df_b = pd.read_csv("../data/processed/dataframes/df_b_zengin_2024_bugun_v2.csv")
df_b["tarih"] = pd.to_datetime(df_b["referans_ayi"])

print("Boyut (satir, sutun):", df_b.shape)
print("Kapsadigi tarih araligi:", df_b["referans_ayi"].min(), "->", df_b["referans_ayi"].max())


**Bu sonuç bize ne söylüyor?**

DF-B, 30 satır (ay) ve 25 sütundan oluşuyor — DF-A'nın 102 satırına göre
çok daha kısa bir pencere, ama 9 tane DF-A'da hiç olmayan yeni sütun
(BETAM + ENAG) içeriyor.


In [ ]:
df_b.head()


**Bu sonuç bize ne söylüyor?** İlk satırlar 2024-01'den başlıyor —
DF-A'nın 2018-01 başlangıcından çok daha yakın bir tarih. Sütunlara
bakınca DF-A'daki tanıdık isimlerin (kur, TÜFE, faiz, ODMD, OSD, noter
devri) yanında `proxy_` ve `enag_` önekli yeni sütunlar da görüyorsun —
bunları Bölüm 3'te tanıyacağız.


---
## Bölüm 2 — 📖 Temel Kavram: Neden İki Ayrı Tablo Kullanıyoruz?

Düşün ki bir arkadaşının **son 30 gününü** çok ayrıntılı biliyorsun —
her gün ne yedi, nasıl hissetti, kiminle konuştu. Ama 10 yıl önce nasıl
biriydi, pek bilmiyorsun. Başka bir arkadaşının ise **son 10 yılını**
genel hatlarıyla biliyorsun (okulu bitirdi, iş değiştirdi, taşındı) ama
son 30 gününde tam olarak ne yaşadığını bilmiyorsun.

İkisi de değerli bilgi — sadece FARKLI SORULARA cevap veriyorlar:
- "Bu kişi genel olarak nasıl bir insan, uzun vadede nasıl değişti?"
  sorusuna **10 yıllık genel bilgi** (bizim DF-A'mız gibi) cevap verir.
- "Şu an tam olarak neler oluyor, hangi detaylar önemli?" sorusuna
  **son 30 günün ayrıntısı** (bizim DF-B'miz gibi) cevap verir.

**Projemize faydası:** DF-A (uzun ama dar) ile "genel eğilim/mevsimsellik
ne?" sorusunu; DF-B (kısa ama zengin) ile "acaba ikinci el fiyatı ve
piyasa hızı, noter devriyle ilişkili mi?" sorusunu cevaplıyoruz. Bu
notebook'ta ikinci soruyla ilgileneceğiz.


---
## Bölüm 3 — Yeni Sütunlarla Tanışma: BETAM ve ENAG

DF-A ile ortak olan sütunları (kur, TÜFE, faiz, ODMD, OSD, tüketici
güveni, noter devri) burada TEKRAR anlatmayacağız — onları DF-A
notebook'unda görmüştün. Burada yalnızca DF-B'YE ÖZGÜ, yeni sütunları
ele alıyoruz.


### BETAM'ın üç ham göstergesi

**BETAM nedir?** Bahçeşehir Üniversitesi'ne bağlı bir araştırma merkezi;
sahibinden.com'daki otomobil ilanlarını analiz ederek aylık "Otomobil
Piyasası Görünümü" raporları yayımlıyor. Bu raporlardan üç göstergeyi
kullanıyoruz:


In [ ]:
print(df_b["proxy_fiyat_cari_tl"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_b["tarih"], df_b["proxy_fiyat_cari_tl"], marker="o", markersize=3)
# Eksik olan 2 ayi (2024-05, 2025-02) kirmizi cizgilerle isaretleyelim
for eksik_ay in df_b.loc[df_b["proxy_fiyat_cari_tl"].isna(), "tarih"]:
    plt.axvline(eksik_ay, color="red", linestyle="--", alpha=0.5)
plt.title("proxy_fiyat_cari_tl - zaman icinde seyri (kirmizi kesikli = veri yok)")
plt.xlabel("Tarih")
plt.ylabel("TL")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** `proxy_fiyat_cari_tl`, ikinci el
otomobil piyasasındaki ORTALAMA İLAN FİYATINI (TL, mix/kompozisyon
düzeltmesi yapılmamış ham ortalama) gösteriyor. Ortalama ~990 bin TL,
min 856 bin TL, max 1 milyon 175 bin TL. **28/30 ay dolu** — grafikte
kırmızı kesikli çizgilerle işaretlenen **2024-05 ve 2025-02** ayları
BOŞ, çünkü BETAM o iki ay için hiç rapor yayımlamadı (bizim bir hatamız
değil, kaynağın kendi boşluğu).


In [ ]:
print(df_b["proxy_dom_gun"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_b["tarih"], df_b["proxy_dom_gun"], marker="o", markersize=3, color="darkorange")
for eksik_ay in df_b.loc[df_b["proxy_dom_gun"].isna(), "tarih"]:
    plt.axvline(eksik_ay, color="red", linestyle="--", alpha=0.5)
plt.title("proxy_dom_gun - zaman icinde seyri (kirmizi kesikli = veri yok)")
plt.xlabel("Tarih")
plt.ylabel("Gun")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** `proxy_dom_gun` (İngilizce "days on
market"tan kısaltma), bir ilanın ortalama kaç GÜN piyasada kaldığını
(yani satılana kadar geçen süreyi) gösteriyor. Ortalama ~22 gün, min 19,1
max 25,6 gün — nispeten istikrarlı bir seri (std sapma yalnızca 1,76 gün).
Aynı 2 ay (2024-05, 2025-02) burada da boş, çünkü aynı BETAM raporundan
geliyor.


In [ ]:
print(df_b["proxy_satis_orani_pct"].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_b["tarih"], df_b["proxy_satis_orani_pct"], marker="o", markersize=3, color="green")
for eksik_ay in df_b.loc[df_b["proxy_satis_orani_pct"].isna(), "tarih"]:
    plt.axvline(eksik_ay, color="red", linestyle="--", alpha=0.5)
plt.title("proxy_satis_orani_pct - zaman icinde seyri (kirmizi kesikli = veri yok)")
plt.xlabel("Tarih")
plt.ylabel("%")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** `proxy_satis_orani_pct`, ilana çıkan
otomobillerin YÜZDE KAÇININ fiilen satışla sonuçlandığını gösteriyor.
Ortalama %21, min %14,9, max %25,5 — üç BETAM göstergesi içinde en
OYNAK olanı (std sapma 2,37 puan, diğerlerine göre görece büyük). Aynı
2 ay yine boş.


### Fiyattan türetilen 4 değişim sütunu

`proxy_fiyat_cari_tl`'den 4 farklı "aylık değişim" ölçüsü türetilmiş:
iki tanesi NOMİNAL (ham TL fiyatının değişimi), iki tanesi REEL (TÜFE'ye
bölünerek enflasyondan arındırılmış fiyatın değişimi) — her ikisinde de
hem yüzde-değişim hem log-değişim versiyonu var (DF-A'da öğrendiğin gibi,
küçük değişimlerde ikisi hemen hemen aynı sonucu verir).


In [ ]:
print(df_b[["proxy_nominal_aylik_pct", "proxy_reel_aylik_pct",
            "proxy_aylik_log_degisim", "proxy_reel_aylik_log_degisim"]].describe())


In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(df_b["tarih"], df_b["proxy_aylik_log_degisim"], marker="o", markersize=3, label="Nominal (ham TL)")
plt.plot(df_b["tarih"], df_b["proxy_reel_aylik_log_degisim"], marker="o", markersize=3, label="Reel (TUFE'den aritilmis)")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Proxy Fiyatin Aylik Log-Degisimi: Nominal vs Reel")
plt.xlabel("Tarih")
plt.ylabel("Log-degisim")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** Nominal ortalama +1,16 (yani ham TL
fiyatı genelde AY-AY ARTIYOR), reel ortalama ise **-1,39** (yani
enflasyondan arındırılınca fiyat genelde AY-AY AZALIYOR). Bu ilk bakışta
çelişkili görünebilir ama aslında mantıklı: TL enflasyonu o kadar
yüksek ki, ham fiyat TL cinsinden artsa bile, bu artış çoğu zaman genel
fiyat artışının GERİSİNDE kalıyor — yani "gerçek" (satın alma gücü
cinsinden) fiyat aslında geriliyor. Bu, projenin daha önce de not ettiği
önemli bir bulgu: nominal ve reel bakış açısı burada TAMAMEN FARKLI bir
hikaye anlatıyor.


### ENAG'ın enflasyon ölçümü

**ENAG nedir?** Enflasyon Araştırma Grubu — TÜİK'in resmi TÜFE
rakamlarına BAĞIMSIZ bir alternatif enflasyon ölçümü sunan bir grup.
İki farklı kurum, iki farklı yöntemle aynı şeyi (fiyatların ne kadar
arttığını) ölçmeye çalışıyor.


In [ ]:
print(df_b[["enag_aylik", "enag_yillik"]].describe())


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_b["tarih"], df_b["enag_yillik"], marker="o", markersize=3, color="purple")
plt.title("ENAG Yillik Enflasyon Olcumu - zaman icinde seyri")
plt.xlabel("Tarih")
plt.ylabel("%")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** ENAG'ın yıllık enflasyon ölçümü,
dönem boyunca %51,5 ile %129,1 arasında değişmiş — yüksek ama zamanla
azalan bir seyir izliyor. Bunu TÜİK'in kendi rakamıyla nasıl
karşılaştıracağımızı Bölüm 5'te detaylı ele alacağız.


---
## Bölüm 4 — 📖 Temel Kavram: Az Gözlemle Çalışmanın Riski

Bir parayı 3 kere havaya attın, 3'ü de yazı geldi. Bunu görüp "bu para
HER ZAMAN yazı gelir" diyebilir misin? Hayır — çünkü çok az deneme
yaptın, bu 3 yazı tamamen tesadüf olabilir. Parayı 1000 kere atsan ve
yine çoğunlukla yazı gelseydi, o zaman "bu para gerçekten yazıya meyilli"
demek çok daha güvenilir olurdu.

> 📖 **Terim — Gözlem Sayısı (n):** Bir analizde kullanılan veri
> noktalarının (bizim durumumuzda: kaç AYIN verisinin) sayısı. Genel
> kural: **n arttıkça, bulduğun ilişkilere/sonuçlara olan güvenin de
> artar.** n küçükse (özellikle birkaç düzineden az), tesadüfen ortaya
> çıkan "sahte" bir örüntüyü gerçek bir ilişki sanma riski YÜKSEKTİR.

**DF-B'nin durumu:** DF-B yalnızca **30 satır (ay)** içeriyor — DF-A'nın
102 satırına göre çok daha az. Üstelik BETAM'ın 2 ay veri
vermediği ayları (2024-05, 2025-02) çıkarınca ve bir de "bu ay geçen
aya göre ne kadar değişti" hesaplarken ilk ayı kaybedince, elimizde
kalan GERÇEKTEN kullanılabilir gözlem sayısı 24-28 arasına düşüyor.

**Bu ne anlama geliyor?** Bölüm 6'da göreceğimiz korelasyon sonuçlarını
DF-A'daki (102 gözlemli) sonuçlardan DAHA TEMKİNLİ okumalıyız — yüksek
bir r değeri görsek bile, bu ~25-30 gözlemlik bir pencerede tesadüfen
ortaya çıkmış olabilir. "Kesin kanıt" değil, "izlenmeye değer sinyal"
diye düşünmek daha doğru.


In [ ]:
print("DF-B toplam satir sayisi:", len(df_b))
print("BETAM sutunlarinda dolu ay sayisi:", df_b["proxy_fiyat_cari_tl"].notna().sum())
print("Aylik degisim hesaplandiktan sonra gecerli gozlem sayisi (proxy_aylik_log_degisim):",
      df_b["proxy_aylik_log_degisim"].notna().sum())


**Bu sonuç bize ne söylüyor?** 30 ayla başlıyoruz, BETAM'ın 2 boşluğu
yüzünden 28'e, "aylık değişim" hesaplaması yüzünden de (ilk ay + boşluğa
komşu aylar kaybolduğu için) 25'e düşüyoruz. Bölüm 6'daki her korelasyon
sonucunu bu n=~24-28 penceresiyle okumalıyız.


---
## Bölüm 5 — İki Bağımsız Enflasyon Ölçümü: TÜİK ve ENAG

TÜİK (Türkiye İstatistik Kurumu), Türkiye'nin RESMİ istatistik
kurumudur ve enflasyonu (TÜFE) kendi metodolojisiyle her ay açıklar.
ENAG ise buna bağımsız bir alternatif sunan sivil bir araştırma
grubudur. İki kurum da aynı soruyu ("fiyatlar ne kadar arttı?") farklı
yöntemlerle cevaplıyor — bu notebook onların hangisinin "doğru"
olduğuna karar vermez, yalnızca ikisini yan yana koyar.


In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df_b["tarih"], df_b["tufe_yillik_degisim"], marker="o", markersize=3, label="TUIK (TUFE) yillik enflasyon")
plt.plot(df_b["tarih"], df_b["enag_yillik"], marker="o", markersize=3, label="ENAG yillik enflasyon")
plt.title("TUIK vs ENAG - Yillik Enflasyon Karsilastirmasi")
plt.xlabel("Tarih")
plt.ylabel("% (yillik)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fark = df_b["enag_yillik"] - df_b["tufe_yillik_degisim"]
print("Fark (ENAG - TUIK) istatistikleri:")
print(f"  Ortalama: {fark.mean():.2f} puan")
print(f"  En genis fark: {fark.max():.2f} puan ({df_b.loc[fark.idxmax(), 'referans_ayi']})")
print(f"  En dar fark: {fark.min():.2f} puan ({df_b.loc[fark.idxmin(), 'referans_ayi']})")


**Bu sonuç bize ne söylüyor?** İki ölçüm arasındaki fark, dönem
boyunca **64,25 puandan (2024-01) 19,39 puana (2026-06) kadar sürekli
daralmış** — yani ENAG'ın rakamı TÜİK'in rakamına GİDEREK YAKLAŞIYOR
(tamamen aynı değiller ama makas kapanıyor). Bu daralma trendinin NEDEN
olduğu (TÜİK mi ENAG'a yaklaşıyor, ENAG mı yavaşlıyor, yoksa iki
metodoloji arasındaki fark mı zamanla azalıyor) bu notebook'un
kapsamı dışında — yalnızca gözlemi raporluyoruz, yorumunu ekip lideri
yapacak.

**Neden bu ikisi TEK bir sütunda birleştirilmedi?** Çünkü ikisi FARKLI
metodolojilerle üretiliyor — birleştirmek, hangi kaynağın hangi ayda
kullanıldığını gizler ve gelecekte "hangi kaynak ne diyordu" sorusunu
cevaplamayı zorlaştırırdı. Ayrı sütunlar tutmak, istediğin zaman
ikisini ÇAPRAZ KONTROL edebilmeni sağlıyor.


---
## Bölüm 6 — Log-Değişim Korelasyonu (DF-A'daki Yöntemin DF-B'ye Uygulanışı)

**Kısa hatırlatma (detay için DF-A notebook'una bak):** Ham seviye
değerleri (kur, fiyat gibi) yerine "bu ay geçen aya göre ne kadar
değişti"ye (log-değişim) bakmak, iki serinin sırf ikisi de zamanla
büyüdüğü için yapay şekilde yüksek korelasyon göstermesini (sahte
korelasyon) önlüyordu. Aynı yöntemi şimdi DF-B'ye uyguluyoruz.

**Not:** DF-B'de bazı sütunlar (`tufe_aylik_degisim`,
`proxy_aylik_log_degisim`, `enag_aylik` gibi) zaten birer "değişim"
ölçüsü olduğu için onlara TEKRAR log-değişim uygulamıyoruz, olduğu gibi
kullanıyoruz — yalnızca ham SEVİYE olan sütunlar (kur, faiz, ODMD, OSD,
anketler, noter devri, BETAM'ın DOM/satış oranı) için yeni log-değişim
hesaplıyoruz.


In [ ]:
# Seviye tipi sutunlar - yeni log-degisim hesaplanacak
# (tufe_endeks ve proxy_fiyat_cari_tl HARIC - onlarin degisim versiyonlari zaten var)
seviye_sutunlari = [
    "usdtry_aysonu", "usdtry_ortalama", "tasit_kredisi_faiz", "politika_faizi",
    "odmd_otomobil_adet", "osd_binek_adet", "tuketici_guven_endeksi",
    "otomobil_satinalma_ihtimali_endeksi", "noter_devir_toplam_adet", "noter_devir_otomobil_adet",
    "proxy_dom_gun", "proxy_satis_orani_pct",
]

log_degisim_b = pd.DataFrame()
for sutun in seviye_sutunlari:
    log_degisim_b[sutun] = np.log(df_b[sutun] / df_b[sutun].shift(1))

# Zaten "degisim" olan sutunlari oldugu gibi ekliyoruz (tekrar log almiyoruz)
for sutun in ["tufe_aylik_degisim", "tufe_yillik_degisim",
              "proxy_aylik_log_degisim", "proxy_reel_aylik_log_degisim",
              "enag_aylik", "enag_yillik"]:
    log_degisim_b[sutun] = df_b[sutun]

print("Toplam sutun:", log_degisim_b.shape[1])
print("Tam dolu (gecerli) satir sayisi:", log_degisim_b.dropna().shape[0], "/", len(log_degisim_b))


In [ ]:
korelasyon_b = log_degisim_b.corr()

plt.figure(figsize=(10, 9))
im = plt.imshow(korelasyon_b.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(korelasyon_b.columns)), korelasyon_b.columns, rotation=90, fontsize=8)
plt.yticks(range(len(korelasyon_b.columns)), korelasyon_b.columns, fontsize=8)
plt.colorbar(im, label="Korelasyon (r)")
plt.title("DF-B Log-Degisim Korelasyon Matrisi")
plt.tight_layout()
plt.show()


**Bu sonuç bize ne söylüyor?** En dikkat çekici (|r| > 0,5) birkaç
çift:

| Çift | r |
|---|---|
| noter_devir_toplam_adet ↔ noter_devir_otomobil_adet | 0,994 |
| tufe_yillik_degisim ↔ enag_yillik | 0,971 |
| tufe_aylik_degisim ↔ enag_aylik | 0,894 |
| noter_devir_otomobil_adet ↔ proxy_satis_orani_pct | 0,867 |
| noter_devir_toplam_adet ↔ proxy_satis_orani_pct | 0,830 |
| tufe_endeks-degisim ↔ enag_yillik (ters) | -0,60 civarı |

TÜİK ve ENAG'ın aylık/yıllık değişim ölçümlerinin yüksek korelasyonu
(0,89-0,97) şaşırtıcı değil — Bölüm 5'te de gördüğümüz gibi ikisi de
aynı olguyu (enflasyon) ölçmeye çalışıyor, farklı yöntemle de olsa
BİRLİKTE hareket etmeleri beklenir.


### Noter devrinin diğer TÜM özelliklerle korelasyonu

Şimdi özellikle projenin hedefine (noter devri) odaklanalım.


In [ ]:
diger_sutunlar = [c for c in korelasyon_b.columns
                   if c not in ["noter_devir_toplam_adet", "noter_devir_otomobil_adet"]]

for hedef in ["noter_devir_toplam_adet", "noter_devir_otomobil_adet"]:
    print(f"=== {hedef} ile digerlerinin korelasyonu (buyukten kucuge) ===")
    print(korelasyon_b.loc[hedef, diger_sutunlar].sort_values(key=abs, ascending=False))
    print()


**Bu sonuç bize ne söylüyor?**

Noter devriyle en güçlü ilişki **`proxy_satis_orani_pct`** (BETAM'ın
satış oranı, r≈0,83-0,87) — bu, iki BAĞIMSIZ kaynağın (TÜİK'in noter
devri, BETAM'ın ilan-satış oranı) aynı piyasa hareketliliğini
yakaladığına işaret ediyor, dikkat çekici bir bulgu.

İkinci sırada `odmd_otomobil_adet` (sıfır km satış, r≈0,44) ve
`osd_binek_adet` (yerli üretim, r≈0,43) geliyor — bunlar DF-A
notebook'unda da benzer bir sinyal vermişti (Bölüm 7'de karşılaştıracağız).

**Beklenmedik bir bulgu:** `proxy_aylik_log_degisim` (nominal fiyat
değişimi) ile noter devri arasında NEGATİF bir ilişki var (r≈-0,53/-0,55)
— yani fiyat AY-AY hızlı arttığında, noter devri (işlem hacmi) genelde
AZALMA eğiliminde. Bu sezgiye aykırı görünebilir (normalde "fiyat
artıyorsa piyasa canlı" diye düşünülür) ama az-gözlem uyarısını
(Bölüm 4) unutmayalım — bu **n≈24-25 gözlemle** hesaplandı, kesin bir
kanıt değil, ekip lideriyle tartışılmaya değer bir sinyal.


---
## Bölüm 7 — Stratejik Çıkarımlar

**Not:** Bu bölüm bir KARAR belgesi değil. Hangi özelliğin kullanılacağı,
model kurulup kurulmayacağı gibi kararlar proje sahibine ait — burada
yalnızca bulguları özetliyoruz.

### DF-A ile DF-B tutarlı mı?

DF-A ve DF-B'de ORTAK olan iki özelliğin (odmd_otomobil_adet,
osd_binek_adet) noter devriyle olan ilişkisini karşılaştıralım:

| Özellik | DF-A'da r (n=100) | DF-B'de r (n=24) | Tutarlı mı? |
|---|---|---|---|
| odmd_otomobil_adet | ≈0,46 | ≈0,44 | ✅ Çok tutarlı — neredeyse aynı büyüklükte, aynı yönde |
| osd_binek_adet | ≈0,60 | ≈0,43 | ⚠️ Aynı yönde (pozitif) ama büyüklük farklı — muhtemelen küçük örneklem (DF-B) farkı |

**Bu ne anlama geliyor?** `odmd_otomobil_adet`'in iki FARKLI tabloda,
FARKLI zaman pencerelerinde benzer bir ilişki göstermesi, bunun rastgele
bir tesadüf olmadığını, gerçek bir sinyal olabileceğini düşündürüyor —
bu, projenin GÜVEN DÜZEYİNİ artıran bir bulgu. `osd_binek_adet`'in
büyüklüğü değişse de yönü (pozitif) tutarlı kalmış.

### Bu zengin ama kısa veriyle neler yapılabilir?

1. **`proxy_satis_orani_pct`, noter devriyle en güçlü ilişkiyi gösteren
   özellik** (r≈0,83-0,87) — ama bu yalnızca DF-B'de test edilebiliyor
   (BETAM verisi DF-A'da yok), bu yüzden DF-A ile çapraz doğrulanamıyor.
2. **Fiyat değişimi (nominal) ile noter devri arasındaki NEGATİF ilişki**
   ilginç ve daha derin araştırmaya değer bir bulgu — ama az gözlemle
   (n≈24-25) hesaplandığı için temkinli okunmalı.
3. **TÜİK-ENAG farkının daralması**, ekonomideki genel bir "yavaşlama/
   normalleşme" sinyali olabilir — bu, doğrudan noter devriyle
   ilişkilendirilmedi ama ayrı bir izleme konusu olarak akılda tutulabilir.

### Olası sonraki adımlar (yalnızca öneri, karar değil)

- DF-B'nin ay sayısı arttıkça (zaman geçtikçe), bu korelasyonlar daha
  büyük bir örneklemle yeniden hesaplanıp doğrulanabilir.
- `proxy_satis_orani_pct` ile noter devri arasındaki ilişki, GECİKMELİ
  (bu ayki satış oranı, gelecek ayki devri mi etkiliyor?) olarak da
  test edilebilir.
- Nominal fiyat değişimi ile noter devri arasındaki ters ilişkinin
  ekonomik bir açıklaması olup olmadığı (örneğin "fiyat hızlı artınca
  alıcılar bekliyor" gibi bir hipotez) ekip lideriyle tartışılabilir.
